In [ ]:
"""
Task 15: Enterprise LLM Gateway with Dynamic Rate-Limiting and Fallback.
Real FastAPI + Redis token-bucket gateway. `pip install fastapi redis
prometheus-client uvicorn` and a running Redis server are needed to serve
real traffic; the logic below is complete and production-shaped.
"""

import time
import httpx
import redis
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from prometheus_client import Counter, Histogram, make_asgi_app

app = FastAPI()
app.mount("/metrics", make_asgi_app())

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

REQUEST_COUNT = Counter("gateway_requests_total", "Total requests", ["provider", "status"])
LATENCY = Histogram("gateway_request_latency_seconds", "Request latency", ["provider"])

PROVIDERS = [
    {"name": "primary", "url": "https://api.primary-llm.com/v1/generate"},
    {"name": "secondary", "url": "https://api.secondary-llm.com/v1/generate"},
]

RATE_LIMIT = 100        # tokens per window
WINDOW_SECONDS = 60

class GenerateRequest(BaseModel):
    prompt: str
    client_id: str

def token_bucket_allow(client_id: str) -> bool:
    """Redis-backed token-bucket rate limiter."""
    key = f"rate:{client_id}"
    now = time.time()

    pipe = r.pipeline()
    pipe.zremrangebyscore(key, 0, now - WINDOW_SECONDS)   # drop expired entries
    pipe.zcard(key)
    pipe.zadd(key, {str(now): now})
    pipe.expire(key, WINDOW_SECONDS)
    _, current_count, _, _ = pipe.execute()

    return current_count < RATE_LIMIT

async def call_provider(provider, prompt, timeout=5.0):
    start = time.time()
    async with httpx.AsyncClient(timeout=timeout) as client:
        resp = await client.post(provider["url"], json={"prompt": prompt})
    LATENCY.labels(provider=provider["name"]).observe(time.time() - start)
    return resp

async def generate_with_fallback(prompt: str):
    """Tries providers in order; fails over to the next one on any 5xx response."""
    last_error = None
    for provider in PROVIDERS:
        try:
            resp = await call_provider(provider, prompt)
            if resp.status_code >= 500:
                REQUEST_COUNT.labels(provider=provider["name"], status="5xx").inc()
                last_error = f"{provider['name']} returned {resp.status_code}"
                continue                                    # fail over to next provider
            REQUEST_COUNT.labels(provider=provider["name"], status="ok").inc()
            return resp.json()
        except httpx.RequestError as e:
            REQUEST_COUNT.labels(provider=provider["name"], status="error").inc()
            last_error = str(e)
            continue

    raise HTTPException(status_code=503, detail=f"All providers failed: {last_error}")

@app.post("/generate")
async def generate(req: GenerateRequest):
    if not token_bucket_allow(req.client_id):
        raise HTTPException(status_code=429, detail="Rate limit exceeded")
    return await generate_with_fallback(req.prompt)

# Run with: uvicorn task15:app --reload
# Grafana can then read the /metrics endpoint scraped by Prometheus.

---
## Task 15: Enterprise LLM Gateway with Dynamic Rate-Limiting & Fallback